# Module 03 — Lesson 02
# Variance and Standard Deviation

---

> The mean tells you where the data is. Standard deviation tells you how sure you can be about that.

Consider two cricket batsmen:

| Batsman | Scores | Average |
|---|---|---|
| Rohit | 0, 12, 5, 150, 8, 120, 2 | ~42 |
| Virat | 38, 45, 40, 52, 41, 48, 30 | ~42 |

Same average. Very different players. Rohit is explosive but inconsistent. Virat is a dependable run-machine. **Standard deviation measures that inconsistency.** Rohit's std dev is ~60. Virat's is ~7.

This matters everywhere in data science:
- **Finance:** two stocks with the same expected return but different volatility (std dev = risk)
- **Manufacturing:** a machine producing parts within 0.01mm vs 1mm tolerance
- **ML preprocessing:** before feeding data into a model, you standardise it using mean and std dev — that's `StandardScaler` in scikit-learn
- **Outlier detection:** a value more than 3 standard deviations from the mean is almost certainly unusual

---
## 🎯 Learning Objectives

By the end of this lesson, you will be able to:

- Explain why mean alone is not enough to describe a dataset
- Compute population and sample variance from scratch, and know when each applies
- Compute standard deviation and interpret it in plain English
- Use the Coefficient of Variation to compare spread across datasets with different scales
- Compute z-scores to standardise data and detect outliers
- Understand why z-score standardisation is essential before ML modelling

---
## 1. Why Mean Alone Isn't Enough

In [ ]:
import math

def mean(data):
    if not data:
        raise ValueError("Cannot compute mean of an empty list")
    return sum(data) / len(data)

# Two classes — same mean, very different picture
class_a = [73, 74, 75, 75, 76, 76, 77, 74, 75, 75]  # everyone clustered near 75
class_b = [50, 50, 55, 60, 100, 100, 95, 90, 50, 50] # split into two groups

print(f"Class A scores : {class_a}")
print(f"Class A mean   : {mean(class_a):.1f}")
print()
print(f"Class B scores : {class_b}")
print(f"Class B mean   : {mean(class_b):.1f}")
print()
print("Same mean (75.0). But Class A is consistent; Class B is polarised.")
print("The mean cannot distinguish them. We need a measure of SPREAD.")

In [ ]:
# Visualise the spread with an ASCII dot plot
def dot_plot(data, label, width=60, lo=40, hi=110):
    """ASCII dot plot — each value placed on a number line."""
    line = ["-"] * width
    for v in data:
        pos = int((v - lo) / (hi - lo) * (width - 1))
        pos = max(0, min(width - 1, pos))
        line[pos] = "●"
    # Mark the mean
    m_pos = int((mean(data) - lo) / (hi - lo) * (width - 1))
    if 0 <= m_pos < width:
        line[m_pos] = "▲"
    print(f"  {label}")
    print(f"  |{''.join(line)}|")
    print(f"  {lo}{' '*(width//2 - 4)}mean={mean(data):.0f}{' '*(width//2 - 4)}{hi}")
    print()

dot_plot(class_a, "Class A (consistent)")
dot_plot(class_b, "Class B (polarised)")
print("▲ = mean position   ● = student score")

---
## 2. Deviation from the Mean

The first step toward measuring spread: compute how far each value is from the mean.

$$\text{deviation}_i = x_i - \bar{x}$$

Positive deviation → value is above the mean. Negative → below. But if we average the deviations, they always cancel out to exactly zero — positive and negative deviations balance.

In [ ]:
scores = [60, 70, 75, 80, 90]
m = mean(scores)

deviations = [x - m for x in scores]

print(f"Scores         : {scores}")
print(f"Mean           : {m}")
print()
print(f"  {'Score':>7} {'Deviation (x - mean)':>22}")
print("  " + "─" * 32)
for x, d in zip(scores, deviations):
    print(f"  {x:>7}   {d:>+10.1f}")
print(f"  {'Sum':>7}   {sum(deviations):>+10.1f}  ← always zero")
print()
print("Positive and negative deviations cancel out.")
print("Averaging them gives 0 — useless as a spread measure.")
print("Solution: square each deviation before averaging.")

---
## 3. Variance

Squaring the deviations eliminates the cancellation problem:

### Population Variance (when you have ALL data)
$$\sigma^2 = \frac{\sum_{i=1}^{N}(x_i - \mu)^2}{N}$$

### Sample Variance (when your data is a sample from a larger population)
$$s^2 = \frac{\sum_{i=1}^{n}(x_i - \bar{x})^2}{n - 1}$$

The `n-1` denominator is called **Bessel's correction**. It compensates for the fact that a sample tends to underestimate the true population spread. In practice: if your data is a sample (which it almost always is), use `n-1`.

In [ ]:
def variance(data, population=False):
    """
    Compute variance.
    population=True  → divide by N   (you have ALL the data)
    population=False → divide by N-1 (your data is a sample)  [default]
    """
    if not data:
        raise ValueError("Cannot compute variance of an empty list")
    if len(data) < 2:
        raise ValueError("Need at least 2 data points")
    m          = mean(data)
    sq_devs    = [(x - m) ** 2 for x in data]
    denominator = len(data) if population else len(data) - 1
    return sum(sq_devs) / denominator


scores = [60, 70, 75, 80, 90]
m      = mean(scores)

print(f"Scores : {scores}")
print(f"Mean   : {m}")
print()
print(f"  {'Score':>7} {'Deviation':>11} {'Deviation²':>12}")
print("  " + "─" * 34)
for x in scores:
    d  = x - m
    d2 = d ** 2
    print(f"  {x:>7}   {d:>+9.1f}   {d2:>10.1f}")

sq_devs = [(x - m) ** 2 for x in scores]
print(f"  {'Sum of squares':>18}:  {sum(sq_devs):.1f}")
print()
print(f"  Population variance (÷{len(scores)})  : {variance(scores, population=True):.2f}")
print(f"  Sample variance     (÷{len(scores)-1})  : {variance(scores, population=False):.2f}")
print()
print("Note: variance is in SQUARED units (score²) — hard to interpret directly.")
print("Standard deviation brings it back to the original units.")

In [ ]:
# Compare variance for our two classes
class_a = [73, 74, 75, 75, 76, 76, 77, 74, 75, 75]
class_b = [50, 50, 55, 60, 100, 100, 95, 90, 50, 50]

var_a = variance(class_a)
var_b = variance(class_b)

print(f"Class A — mean={mean(class_a):.1f}, variance={var_a:.2f}")
print(f"Class B — mean={mean(class_b):.1f}, variance={var_b:.2f}")
print()
print(f"Class B variance is {var_b/var_a:.0f}× larger — much more spread out.")

---
## 4. Standard Deviation

Variance has a unit problem — if scores are in points, variance is in points². Taking the square root brings us back to the original units:

$$\sigma = \sqrt{\sigma^2} \quad \text{(population)}$$
$$s = \sqrt{s^2} \quad \text{(sample)}$$

**How to interpret it:** Standard deviation is roughly "the typical distance of a data point from the mean."

In [ ]:
import math

def std_dev(data, population=False):
    """Compute standard deviation (square root of variance)."""
    return math.sqrt(variance(data, population=population))


class_a = [73, 74, 75, 75, 76, 76, 77, 74, 75, 75]
class_b = [50, 50, 55, 60, 100, 100, 95, 90, 50, 50]

for label, data in [("Class A", class_a), ("Class B", class_b)]:
    m   = mean(data)
    var = variance(data)
    std = std_dev(data)
    print(f"{label}:")
    print(f"  Mean               : {m:.1f} points")
    print(f"  Variance           : {var:.2f} points²  ← hard to interpret")
    print(f"  Std deviation      : {std:.2f} points   ← back to original units")
    print(f"  Interpretation     : A typical score is ±{std:.1f} points from the mean of {m:.0f}")
    print()

In [ ]:
# The 68-95-99.7 Rule (Empirical Rule)
# For normally distributed data (bell curve — covered fully in Lesson 06):
#
#   ≈68% of data falls within 1 std dev of the mean  (mean ± 1σ)
#   ≈95% of data falls within 2 std devs             (mean ± 2σ)
#   ≈99.7% of data falls within 3 std devs           (mean ± 3σ)
#
# Anything beyond ±3σ is extremely rare — likely an outlier.

# Exam scores — roughly normal
exam_scores = [65, 70, 72, 73, 74, 75, 75, 76, 76, 77,
               78, 78, 79, 80, 80, 81, 82, 83, 85, 88]

m   = mean(exam_scores)
std = std_dev(exam_scores)

within_1 = sum(1 for x in exam_scores if abs(x - m) <= 1 * std)
within_2 = sum(1 for x in exam_scores if abs(x - m) <= 2 * std)
within_3 = sum(1 for x in exam_scores if abs(x - m) <= 3 * std)
n        = len(exam_scores)

print(f"Exam scores  — mean={m:.1f}, std={std:.1f}")
print(f"\n  Range            Boundary           Count   %")
print("  " + "─" * 55)
print(f"  mean ± 1σ   [{m-std:>6.1f}, {m+std:>6.1f}]   {within_1:>5}   {within_1/n*100:.0f}%  (expect ~68%)")
print(f"  mean ± 2σ   [{m-2*std:>6.1f}, {m+2*std:>6.1f}]   {within_2:>5}   {within_2/n*100:.0f}%  (expect ~95%)")
print(f"  mean ± 3σ   [{m-3*std:>6.1f}, {m+3*std:>6.1f}]   {within_3:>5}   {within_3/n*100:.0f}%  (expect ~99.7%)")

---
## 5. Coefficient of Variation (CV) — Comparing Spread Across Scales

Standard deviation is in the same units as the data. That makes it impossible to compare spread between datasets with different scales — you can't directly compare the std dev of exam scores (0–100) with the std dev of salaries (₹20,000–₹2,00,000).

The **Coefficient of Variation** normalises std dev as a percentage of the mean:

$$CV = \frac{s}{\bar{x}} \times 100\%$$

Lower CV → more consistent. Higher CV → more variable.

In [ ]:
def cv(data):
    """Coefficient of Variation — std dev as % of mean."""
    m = mean(data)
    if m == 0:
        raise ValueError("CV is undefined when mean is zero")
    return (std_dev(data) / m) * 100


# Cricket: who is the more consistent batsman?
rohit  = [0, 12, 5, 150, 8, 120, 2, 65, 140, 3, 50, 90]
virat  = [38, 45, 40, 52, 41, 48, 55, 37, 44, 50, 43, 47]
dhoni  = [30, 55, 28, 70, 40, 62, 20, 75, 33, 58, 45, 50]

print("Batsman Consistency Analysis")
print(f"{'Batsman':<10} {'Mean':>8} {'Std Dev':>10} {'CV':>8}  {'Assessment'}")
print("─" * 60)
for name, scores in [("Rohit", rohit), ("Virat", virat), ("Dhoni", dhoni)]:
    m   = mean(scores)
    std = std_dev(scores)
    c   = cv(scores)
    assessment = "high risk / explosive" if c > 60 else "moderate" if c > 35 else "very consistent"
    print(f"{name:<10} {m:>8.1f} {std:>10.1f} {c:>7.1f}%  {assessment}")

In [ ]:
# Investment risk: comparing two mutual funds
# Both funds have similar average monthly returns — but different risk

fund_a_returns = [2.1, 2.3, 1.9, 2.4, 2.0, 2.2, 1.8, 2.5, 2.1, 2.3, 2.0, 2.2]  # stable
fund_b_returns = [-3.0, 5.5, -1.5, 8.2, -2.0, 6.0, 1.0, -4.5, 7.0, 3.5, -2.5, 5.3]  # volatile

print("Investment Fund Comparison (monthly returns %)")
print(f"{'Fund':<10} {'Avg Return':>12} {'Std Dev':>10} {'CV':>8}  {'Risk Profile'}")
print("─" * 60)
for name, returns in [("Fund A", fund_a_returns), ("Fund B", fund_b_returns)]:
    m   = mean(returns)
    std = std_dev(returns)
    c   = abs(cv(returns))  # use abs — CV can be negative if mean is negative
    risk = "Low risk (stable)" if std < 1 else "High risk (volatile)"
    print(f"{name:<10} {m:>11.2f}% {std:>10.2f} {c:>7.1f}%  {risk}")

print()
print("Fund A is the safer long-term investment (low std dev).")
print("Fund B might feel attractive from the mean alone — std dev reveals the risk.")

---
## 6. Z-Score (Standardisation)

A **z-score** answers: *"How many standard deviations is this value from the mean?"*

$$z_i = \frac{x_i - \bar{x}}{s}$$

Z-scores transform data so that:
- The mean becomes **0**
- The standard deviation becomes **1**

This is called **standardisation** (or z-score normalisation). It's the foundation of `StandardScaler` in scikit-learn — applied to almost every ML model before training.

In [ ]:
def z_scores(data):
    """Compute z-score for every value in data."""
    m   = mean(data)
    std = std_dev(data)
    if std == 0:
        raise ValueError("Cannot compute z-scores: std dev is 0 (all values identical)")
    return [(x - m) / std for x in data]


# Priya scored 88 on a difficult exam. How impressive is that?
# Context: class mean = 65, std dev = 12

class_scores = [55, 58, 60, 62, 63, 65, 65, 66, 68, 70,
                72, 74, 75, 78, 80, 82, 88, 90, 52, 69]

m    = mean(class_scores)
std  = std_dev(class_scores)
priya_score = 88
priya_z     = (priya_score - m) / std

print(f"Class mean : {m:.1f}")
print(f"Class std  : {std:.1f}")
print(f"Priya score: {priya_score}")
print(f"Priya z    : {priya_z:+.2f}")
print()
print(f"Priya is {priya_z:.1f} standard deviations above the mean.")
print("That means she outperformed roughly 97% of her class (z≈+1.8 is top ~3%).")

In [ ]:
# Computing z-scores for the full class
zs = z_scores(class_scores)

print("Class Z-scores:")
print(f"  {'Score':>7} {'Z-score':>9}  {'Category'}")
print("  " + "─" * 38)
for score, z in sorted(zip(class_scores, zs), reverse=True):
    if z > 2:
        cat = "★ Exceptional"
    elif z > 1:
        cat = "↑ Above average"
    elif z > -1:
        cat = "  Average range"
    elif z > -2:
        cat = "↓ Below average"
    else:
        cat = "⚠ Needs support"
    print(f"  {score:>7}   {z:>+7.2f}  {cat}")

In [ ]:
# After standardisation: verify mean=0, std=1
zs = z_scores(class_scores)
z_mean = mean(zs)
z_std  = std_dev(zs)

print(f"Original — mean={mean(class_scores):.1f}, std={std_dev(class_scores):.1f}")
print(f"Z-scores — mean={z_mean:.10f} (≈ 0), std={z_std:.10f} (≈ 1)")
print()
print("This is exactly what scikit-learn's StandardScaler does:")
print("  from sklearn.preprocessing import StandardScaler")
print("  scaler = StandardScaler()")
print("  X_scaled = scaler.fit_transform(X)")
print()
print("Why ML models need this: if one feature ranges from 0–100 and another")
print("from 0–1,000,000, the large-scale feature dominates the model.")
print("Z-scoring puts all features on the same scale.")

---
## 7. Outlier Detection with Z-Scores

The **3σ rule**: in a roughly normal distribution, values with |z| > 3 occur less than 0.3% of the time. Such values are almost certainly outliers (data entry errors, sensor faults, genuinely exceptional events).

In [ ]:
def find_outliers_zscore(data, threshold=3.0):
    """
    Identify outliers using the z-score method.
    Returns list of (index, value, z_score) for outliers.
    Default threshold of 3 is the standard choice.
    """
    m   = mean(data)
    std = std_dev(data)
    outliers = []
    for i, x in enumerate(data):
        z = (x - m) / std
        if abs(z) > threshold:
            outliers.append((i, x, round(z, 2)))
    return outliers


# Sensor readings from a temperature logger (°C)
temperatures = [
    22.1, 22.3, 22.0, 22.4, 22.2, 21.9, 22.1, 22.3,
    22.0, 22.2, 85.7,   # ← sensor spike
    22.1, 22.2, 22.3, 22.0, 22.1, -15.3,  # ← sensor dropout
    22.2, 22.1, 22.0
]

outliers = find_outliers_zscore(temperatures, threshold=3.0)

print(f"Temperature sensor — {len(temperatures)} readings")
print(f"Mean  : {mean(temperatures):.2f}°C")
print(f"Std   : {std_dev(temperatures):.2f}°C")
print()
print(f"Outliers (|z| > 3):")
if outliers:
    for idx, val, z in outliers:
        direction = "high" if z > 0 else "low"
        print(f"  Index {idx:>2}: {val:>7.1f}°C  z={z:>+6.2f}  ({direction} spike)")
else:
    print("  None found")

# Clean data by removing outliers
clean_idx  = set(i for i, _, _ in outliers)
clean_temps = [v for i, v in enumerate(temperatures) if i not in clean_idx]
print(f"\nAfter removing outliers:")
print(f"  Mean : {mean(clean_temps):.2f}°C")
print(f"  Std  : {std_dev(clean_temps):.4f}°C")

In [ ]:
# Applied: student score dataset with outlier detection + full stats
def full_stats(data, label="data"):
    """
    Return a complete stats dict: mean, median, std, variance, CV, outliers.
    Builds on Lesson 01's describe() and adds spread metrics.
    """
    from math import sqrt

    def median(d):
        s = sorted(d)
        n = len(s)
        return s[n//2] if n % 2 else (s[n//2-1] + s[n//2]) / 2

    def mode(d):
        freq = {}
        for x in d: freq[x] = freq.get(x, 0) + 1
        mx = max(freq.values())
        return sorted(k for k, v in freq.items() if v == mx)

    m   = mean(data)
    med = median(data)
    mod = mode(data)
    var = variance(data)
    std = sqrt(var)
    coefficient_var = (std / m * 100) if m != 0 else None
    outlier_list    = find_outliers_zscore(data, threshold=2.5)

    print(f"── {label} ──")
    print(f"  n        : {len(data)}")
    print(f"  mean     : {m:.2f}")
    print(f"  median   : {med}")
    print(f"  mode     : {mod}")
    print(f"  variance : {var:.2f}")
    print(f"  std dev  : {std:.2f}")
    print(f"  CV       : {coefficient_var:.1f}%" if coefficient_var else "  CV       : N/A")
    print(f"  outliers : {[(v, z) for _, v, z in outlier_list] or 'none'}")


maths_scores   = [72, 85, 68, 91, 76, 88, 55, 79, 83, 67, 90, 74, 62, 88, 95, 30]
science_scores = [65, 78, 82, 70, 88, 61, 74, 90, 55, 77, 83, 69, 94, 72, 80, 68]

full_stats(maths_scores,   label="Maths")
print()
full_stats(science_scores, label="Science")

---
## 8. Built-in Verification

Python's `statistics` module has these functions built in. After learning the math from scratch, use the built-ins in real code.

In [ ]:
import statistics

data = [72, 85, 68, 91, 76, 88, 55, 79, 83, 67]

print("Verification against statistics module:")
print(f"  {'Measure':<22} {'Ours':>10} {'stdlib':>10} {'Match?'}")
print("  " + "─" * 50)

checks = [
    ("variance (sample)",   variance(data, False),     statistics.variance(data)),
    ("variance (pop)",      variance(data, True),      statistics.pvariance(data)),
    ("std dev (sample)",    std_dev(data, False),      statistics.stdev(data)),
    ("std dev (pop)",       std_dev(data, True),       statistics.pstdev(data)),
]
for label, ours, lib in checks:
    match = abs(ours - lib) < 1e-9
    print(f"  {label:<22} {ours:>10.4f} {lib:>10.4f} {'✓' if match else '✗'}")

---
## ⚠️ Common Mistakes

In [ ]:
# ── Mistake 1: Population vs sample — using wrong denominator ─────────────────

data = [10, 20, 30, 40, 50]

pop_var    = variance(data, population=True)
sample_var = variance(data, population=False)

print("Mistake 1 — Population vs Sample:")
print(f"  Population variance (÷{len(data)})  : {pop_var:.2f}  ← use when you have ALL data")
print(f"  Sample variance     (÷{len(data)-1}) : {sample_var:.2f}  ← use when data is a sample")
print(f"  Difference          : {sample_var - pop_var:.2f}")
print()
print("  In data science, your data is almost always a sample.")
print("  Default: use sample variance (n-1). pandas .var() uses n-1 by default.")

In [ ]:
# ── Mistake 2: Interpreting variance directly — units are squared ─────────────

scores = [60, 70, 75, 80, 90]
var    = variance(scores)
std    = std_dev(scores)

print("Mistake 2 — Squared Units:")
print(f"  Scores are in 'points'")
print(f"  Variance = {var:.1f} 'points²'  ← meaningless to say 'spread is 87 points²'")
print(f"  Std dev  = {std:.1f} 'points'   ← meaningful: typical spread is ±{std:.1f} points")
print()
print("  Always report std dev to stakeholders, not variance.")
print("  Variance is used internally for math — e.g., summing variances.")

In [ ]:
# ── Mistake 3: Z-scores change scale, not shape ───────────────────────────────

skewed = [1, 2, 2, 3, 3, 3, 4, 4, 10, 15]   # right-skewed
zs     = z_scores(skewed)

print("Mistake 3 — Z-scores don't fix skew:")
print(f"  Original  : mean={mean(skewed):.2f}, std={std_dev(skewed):.2f}")
print(f"  Z-scored  : mean≈{mean(zs):.8f}, std≈{std_dev(zs):.8f}")
print(f"  Original values : {skewed}")
print(f"  Z values  : {[round(z, 2) for z in zs]}")
print()
print("  The relative gaps are identical — z-scoring only changes the scale,")
print("  not the distribution shape. If data is skewed, it stays skewed.")
print("  To fix skew: use log transform — covered in the Data Cleaning module.")

In [ ]:
# ── Mistake 4: Std dev of 0 — all values identical ───────────────────────────

constant = [5, 5, 5, 5, 5]

print("Mistake 4 — Std dev of zero:")
print(f"  Data: {constant}")
print(f"  Std dev: {std_dev(constant):.4f}")
print()
print("  A feature with std dev = 0 carries zero information.")
print("  In ML: drop constant features — StandardScaler will raise a divide-by-zero error.")
print("  In the z_scores() function above, we raise ValueError for this case.")

---
## ✏️ Practice Exercises

### Exercise 1 — Starter: Same Mean, Different Spread

Two factories produce bolts with target diameter 10mm. Compute variance and std dev for each. Which factory has better quality control, and by how much?

```python
factory_a = [9.9, 10.1, 10.0, 9.8, 10.2, 10.0, 9.9, 10.1, 10.0, 10.0]
factory_b = [8.5, 11.2, 9.0, 10.8, 10.5, 9.3, 11.0, 8.8, 10.7, 10.2]
```

In [ ]:
factory_a = [9.9, 10.1, 10.0, 9.8, 10.2, 10.0, 9.9, 10.1, 10.0, 10.0]
factory_b = [8.5, 11.2, 9.0, 10.8, 10.5, 9.3, 11.0, 8.8, 10.7, 10.2]
# Your code here

### Exercise 2 — Investment Risk Analyser

You're comparing three investment options. For each:
1. Compute mean return, std dev, and CV
2. Rank them by risk (CV) from safest to riskiest
3. Print a recommendation: "If you want stability, choose X. If you can tolerate risk for higher returns, consider Y."

```python
fd_returns     = [6.5, 6.5, 6.5, 6.5, 6.5, 6.5, 6.5, 6.5]   # Fixed deposit
mutual_returns = [8.2, 12.4, 5.6, 14.1, 3.2, 11.8, 9.5, 7.3]  # Equity mutual fund
crypto_returns = [-35, 80, 15, -45, 120, -20, 60, -30]          # Cryptocurrency
```

In [ ]:
fd_returns     = [6.5, 6.5, 6.5, 6.5, 6.5, 6.5, 6.5, 6.5]
mutual_returns = [8.2, 12.4, 5.6, 14.1, 3.2, 11.8, 9.5, 7.3]
crypto_returns = [-35, 80, 15, -45, 120, -20, 60, -30]
# Your code here

### Exercise 3 — Z-Score Grader

A teacher wants to assign grades based on z-scores rather than fixed cut-offs (since different exams have different difficulty levels):

| Z-score | Grade |
|---|---|
| z > 1.5 | A |
| 0.5 < z ≤ 1.5 | B |
| -0.5 < z ≤ 0.5 | C |
| -1.5 < z ≤ -0.5 | D |
| z ≤ -1.5 | F |

Write `grade_on_curve(scores, names)` that prints each student's score, z-score, and curved grade.

In [ ]:
def grade_on_curve(scores, names):
    # Your code here
    pass

names  = ["Priya", "Rohan", "Meera", "Karan", "Ananya", "Dev", "Riya", "Arjun"]
scores = [92, 45, 78, 65, 88, 55, 71, 83]
grade_on_curve(scores, names)

### Exercise 4 — Data Quality Report

Write `data_quality_report(data, label)` that:
1. Computes mean, std dev, CV
2. Detects outliers at both z > 2 and z > 3 thresholds
3. Prints a formatted report with an assessment:
   - CV < 15%: "Low variability — data is consistent"
   - 15% ≤ CV < 35%: "Moderate variability"
   - CV ≥ 35%: "High variability — check for data quality issues"
4. Test on: a clean dataset, and a dataset with an injected data-entry error

In [ ]:
def data_quality_report(data, label):
    # Your code here
    pass

clean_ages  = [21, 22, 20, 23, 21, 22, 24, 20, 21, 23, 22, 21]
typo_ages   = [21, 22, 20, 23, 21, 212, 24, 20, 21, 23, 22, 21]  # 212 is a typo

data_quality_report(clean_ages, "Student Ages (clean)")
print()
data_quality_report(typo_ages,  "Student Ages (with typo)")

### Exercise 5 — (Challenge) Feature Standardiser

In ML, a dataset has multiple features with different scales. Build a `standardise_dataset(records, numeric_fields)` function that:
1. Extracts each numeric field from a list of dicts
2. Computes mean and std dev per field
3. Returns a new list of dicts with z-scored values for all numeric fields
4. Also returns a `scalers` dict: `{field: {"mean": ..., "std": ...}}` so new data can be transformed later
5. Verify: after standardising, each field should have mean ≈ 0 and std ≈ 1

In [ ]:
def standardise_dataset(records, numeric_fields):
    """
    Z-score standardise numeric fields in a list of dicts.
    Returns (standardised_records, scalers).
    scalers = {field: {"mean": ..., "std": ...}}
    """
    # Your code here
    pass


# Dataset: age (20–25), score (50–100), salary (30k–90k) — very different scales
students = [
    {"name": "Priya",  "age": 20, "score": 88, "salary": 85000},
    {"name": "Rohan",  "age": 22, "score": 72, "salary": 62000},
    {"name": "Meera",  "age": 21, "score": 95, "salary": 90000},
    {"name": "Karan",  "age": 23, "score": 61, "salary": 55000},
    {"name": "Ananya", "age": 22, "score": 78, "salary": 70000},
    {"name": "Dev",    "age": 24, "score": 55, "salary": 48000},
]

scaled, scalers = standardise_dataset(students, ["age", "score", "salary"])

if scaled:
    print("Scalers (mean, std per feature):")
    for field, params in scalers.items():
        print(f"  {field:<8}: mean={params['mean']:.1f}, std={params['std']:.2f}")
    print()
    print("Standardised values:")
    for r in scaled:
        print(f"  {r['name']:<8}: age={r['age']:+.2f}  score={r['score']:+.2f}  salary={r['salary']:+.2f}")
    print()
    # Verify
    for field in ["age", "score", "salary"]:
        vals = [r[field] for r in scaled]
        print(f"  {field:<8}: mean={mean(vals):.8f}, std={std_dev(vals):.8f}")

---
## 📌 Key Takeaways

- **Variance and standard deviation measure how spread out data is from the mean.** Two datasets with identical means can look completely different — one tightly clustered, one wildly scattered. Std dev is the number that tells you which situation you're in. A low std dev means the mean is a reliable description of the data; a high std dev means the mean is misleading.

- **Use sample variance (n-1) by default.** In real data science work, your data is almost always a sample from a larger population. Dividing by n-1 (Bessel's correction) compensates for the fact that samples tend to underestimate true population spread. Pandas `.var()` and `.std()` both use n-1 by default (`ddof=1`).

- **Z-scores are the foundation of ML preprocessing.** Before fitting any model, you need features on comparable scales — a salary column (₹0–₹10,00,000) should not dominate an age column (18–60) just because its numbers are larger. Z-scoring transforms every feature to mean=0, std=1, so the model treats all features equally. Understanding the formula `z = (x - mean) / std` tells you exactly what scikit-learn's `StandardScaler` is doing internally.

---
## What's Next?

**Lesson 03 — Percentiles and Correlation**  
You now have two tools: where the data is (mean/median) and how spread out it is (std dev). Next, you'll learn *where any individual value ranks* within a distribution (percentiles — the same as your JEE rank percentile), and whether two variables *move together* (correlation — the backbone of feature selection in ML).